### Load Bronze Tables

In [4]:
from pyspark.sql import functions as F

orders = spark.table("bronze_orders")
order_products_prior = spark.table("bronze_order_products_prior")
order_products_train = spark.table("bronze_order_products_train")
products = spark.table("bronze_products")
aisles = spark.table("bronze_aisles")
departments = spark.table("bronze_departments")


StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 6, Finished, Available, Finished, False)

### Schema Validation

In [5]:
orders.printSchema()
order_products_prior.printSchema()
order_products_train.printSchema()
products.printSchema()
aisles.printSchema()  
departments.printSchema()

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 7, Finished, Available, Finished, False)

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

root
 |-- department_id: integer (nullable = true)
 |-

The data type for aisle_id and department_id in products is string. This needs to be fixed.

In [6]:
products_clean = products.withColumn(
    "aisle_id",
    F.col("aisle_id").cast("int")
).withColumn(
    "department_id",
    F.col("department_id").cast("int")
)

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 8, Finished, Available, Finished, False)

### Register Temporary Views (for SQL)

In [7]:
orders.createOrReplaceTempView("orders")
order_products_prior.createOrReplaceTempView("order_products_prior")
order_products_train.createOrReplaceTempView("order_products_train")
products_clean.createOrReplaceTempView("products")
aisles.createOrReplaceTempView("aisles")
departments.createOrReplaceTempView("departments")

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 9, Finished, Available, Finished, False)

### Data Validation (SQL)

#### Row Count Check

In [8]:
%%sql
SELECT 'orders' AS table_name, COUNT(*) AS rows FROM orders
UNION ALL
SELECT 'order_products_prior', COUNT(*) FROM order_products_prior
UNION ALL
SELECT 'order_products_train', COUNT(*) FROM order_products_train
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'aisles', COUNT(*) FROM aisles
UNION ALL
SELECT 'departments', COUNT(*) FROM departments;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 10, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 2 fields>

#### Null Checks


Orders

In [9]:
%%sql
SELECT
SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id,
SUM(CASE WHEN order_number IS NULL THEN 1 ELSE 0 END) AS null_order_number,
SUM(CASE WHEN order_dow IS NULL THEN 1 ELSE 0 END) AS null_order_dow,
SUM(CASE WHEN order_hour_of_day IS NULL THEN 1 ELSE 0 END) AS null_order_hour,
SUM(CASE WHEN days_since_prior_order IS NULL THEN 1 ELSE 0 END) AS null_days_since_prior
FROM orders;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 6 fields>

Order Products (Prior)

In [10]:
%%sql
SELECT
SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
SUM(CASE WHEN add_to_cart_order IS NULL THEN 1 ELSE 0 END) AS null_add_to_cart_order,
SUM(CASE WHEN reordered IS NULL THEN 1 ELSE 0 END) AS null_reordered
FROM order_products_prior;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

Order Products(Train)

In [11]:
%%sql
SELECT
SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
SUM(CASE WHEN add_to_cart_order IS NULL THEN 1 ELSE 0 END) AS null_add_to_cart_order,
SUM(CASE WHEN reordered IS NULL THEN 1 ELSE 0 END) AS null_reordered
FROM order_products_train;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 13, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

Products

In [12]:
%%sql
SELECT
SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS null_product_name,
SUM(CASE WHEN aisle_id IS NULL THEN 1 ELSE 0 END) AS null_aisle_id,
SUM(CASE WHEN department_id IS NULL THEN 1 ELSE 0 END) AS null_department_id
FROM products;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

Aisles


In [13]:
%%sql 
SELECT
SUM(CASE WHEN aisle_id IS NULL THEN 1 ELSE 0 END) AS null_aisle_id,
SUM(CASE WHEN aisle IS NULL THEN 1 ELSE 0 END) AS null_aisle_name
FROM aisles;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

Departments

In [14]:
%%sql
SELECT
SUM(CASE WHEN department_id IS NULL THEN 1 ELSE 0 END) AS null_department_id,
SUM(CASE WHEN department IS NULL THEN 1 ELSE 0 END) AS null_department_name
FROM departments;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

#### Duplicate Checks

Orders (PK: order_id)

In [15]:
%%sql
SELECT order_id, COUNT(*) AS duplicates
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

Order Products (Prior) (Composite PK)

In [16]:
%%sql
SELECT order_id, product_id, COUNT(*) AS duplicates
FROM order_products_prior
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>

3. Order Products (Train) (Composite PK)

In [17]:
%%sql
SELECT order_id, product_id, COUNT(*) AS duplicates
FROM order_products_train
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>


Products (PK: product_id)

In [18]:
%%sql
SELECT product_id, COUNT(*) AS duplicates
FROM products
GROUP BY product_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

Aisles (PK: aisle_id)

In [19]:
%%sql
SELECT aisle_id, COUNT(*) AS duplicates
FROM aisles
GROUP BY aisle_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

Departments (PK: department_id)

In [20]:
%%sql
SELECT department_id, COUNT(*) AS duplicates
FROM departments
GROUP BY department_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 22, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

#### Referential Integrity

In [21]:
%%sql
SELECT COUNT(*) AS orphan_orders
FROM order_products_prior p
LEFT JOIN orders o
ON p.order_id = o.order_id
WHERE o.order_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [22]:
%%sql
SELECT COUNT(*) AS orphan_products
FROM order_products_prior p
LEFT JOIN products pr
ON p.product_id = pr.product_id
WHERE pr.product_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 24, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [23]:
%%sql
SELECT COUNT(*) AS missing_aisles
FROM products p
LEFT JOIN aisles a
ON p.aisle_id = a.aisle_id
WHERE a.aisle_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [24]:
%%sql
SELECT COUNT(*) AS missing_departments
FROM products p
LEFT JOIN departments d
ON p.department_id = d.department_id
WHERE d.department_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 26, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

#### Range Check

In [25]:
%%sql
SELECT DISTINCT order_dow FROM orders ORDER BY order_dow;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 27, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 1 fields>

In [26]:
%%sql
SELECT DISTINCT order_hour_of_day FROM orders ORDER BY order_hour_of_day;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 28, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 1 fields>

### Exploratory Checks

#### Orders per User

In [27]:
%%sql
SELECT user_id, COUNT(*) AS orders_per_user
FROM orders
GROUP BY user_id
ORDER BY orders_per_user DESC
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 29, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

In [28]:
%%sql
SELECT user_id, COUNT(*) AS orders_per_user
FROM orders
GROUP BY user_id
ORDER BY orders_per_user
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 30, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

#### Basket Size

In [29]:
%%sql
SELECT user_id, COUNT(*) AS orders_per_user
FROM orders
GROUP BY user_id
ORDER BY orders_per_user DESC
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 31, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

In [30]:
%%sql
SELECT order_id, COUNT(product_id) AS basket_size
FROM order_products_prior
GROUP BY order_id
ORDER BY basket_size
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 32, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

#### Reorder Distribution

In [31]:
%%sql
SELECT reordered, COUNT(*) AS count
FROM order_products_prior
GROUP BY reordered;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 33, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

### Creating Product Hierarchy (Denormalization Step)

#### Purpose

This step creates a **denormalized product table** by combining product metadata from multiple tables:

- `products`
- `aisles`
- `departments`



In [32]:
product_details = (
    products_clean
    .join(aisles, "aisle_id", "left")
    .join(departments, "department_id", "left")
)

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 34, Finished, Available, Finished, False)

### Write Silver Tables

### Load Bronze Tables

In [33]:
from pyspark.sql import functions as F

orders = spark.table("bronze_orders")
order_products_prior = spark.table("bronze_order_products_prior")
order_products_train = spark.table("bronze_order_products_train")
products = spark.table("bronze_products")
aisles = spark.table("bronze_aisles")
departments = spark.table("bronze_departments")


StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 35, Finished, Available, Finished, False)

### Schema Validation

In [34]:
orders.printSchema()
order_products_prior.printSchema()
order_products_train.printSchema()
products.printSchema()
aisles.printSchema()  
departments.printSchema()

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 36, Finished, Available, Finished, False)

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

root
 |-- department_id: integer (nullable = true)
 |-

The data type for aisle_id and department_id in products is string. This needs to be fixed.

In [35]:
products_clean = products.withColumn(
    "aisle_id",
    F.col("aisle_id").cast("int")
).withColumn(
    "department_id",
    F.col("department_id").cast("int")
)

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 37, Finished, Available, Finished, False)

### Register Temporary Views (for SQL)

In [36]:
orders.createOrReplaceTempView("orders")
order_products_prior.createOrReplaceTempView("order_products_prior")
order_products_train.createOrReplaceTempView("order_products_train")
products_clean.createOrReplaceTempView("products")
aisles.createOrReplaceTempView("aisles")
departments.createOrReplaceTempView("departments")

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 38, Finished, Available, Finished, False)

### Data Validation (SQL)

#### Row Count Check

In [37]:
%%sql
SELECT 'orders' AS table_name, COUNT(*) AS rows FROM orders
UNION ALL
SELECT 'order_products_prior', COUNT(*) FROM order_products_prior
UNION ALL
SELECT 'order_products_train', COUNT(*) FROM order_products_train
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'aisles', COUNT(*) FROM aisles
UNION ALL
SELECT 'departments', COUNT(*) FROM departments;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 39, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 2 fields>

#### Null Checks


Orders

In [38]:
%%sql
SELECT
SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id,
SUM(CASE WHEN order_number IS NULL THEN 1 ELSE 0 END) AS null_order_number,
SUM(CASE WHEN order_dow IS NULL THEN 1 ELSE 0 END) AS null_order_dow,
SUM(CASE WHEN order_hour_of_day IS NULL THEN 1 ELSE 0 END) AS null_order_hour,
SUM(CASE WHEN days_since_prior_order IS NULL THEN 1 ELSE 0 END) AS null_days_since_prior
FROM orders;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 40, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 6 fields>

Order Products (Prior)

In [39]:
%%sql
SELECT
SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
SUM(CASE WHEN add_to_cart_order IS NULL THEN 1 ELSE 0 END) AS null_add_to_cart_order,
SUM(CASE WHEN reordered IS NULL THEN 1 ELSE 0 END) AS null_reordered
FROM order_products_prior;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 41, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

Order Products(Train)

In [40]:
%%sql
SELECT
SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
SUM(CASE WHEN add_to_cart_order IS NULL THEN 1 ELSE 0 END) AS null_add_to_cart_order,
SUM(CASE WHEN reordered IS NULL THEN 1 ELSE 0 END) AS null_reordered
FROM order_products_train;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 42, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

Products

In [41]:
%%sql
SELECT
SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS null_product_name,
SUM(CASE WHEN aisle_id IS NULL THEN 1 ELSE 0 END) AS null_aisle_id,
SUM(CASE WHEN department_id IS NULL THEN 1 ELSE 0 END) AS null_department_id
FROM products;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 43, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

Aisles


In [42]:
%%sql 
SELECT
SUM(CASE WHEN aisle_id IS NULL THEN 1 ELSE 0 END) AS null_aisle_id,
SUM(CASE WHEN aisle IS NULL THEN 1 ELSE 0 END) AS null_aisle_name
FROM aisles;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 44, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

Departments

In [43]:
%%sql
SELECT
SUM(CASE WHEN department_id IS NULL THEN 1 ELSE 0 END) AS null_department_id,
SUM(CASE WHEN department IS NULL THEN 1 ELSE 0 END) AS null_department_name
FROM departments;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 45, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

#### Duplicate Checks

Orders (PK: order_id)

In [44]:
%%sql
SELECT order_id, COUNT(*) AS duplicates
FROM orders
GROUP BY order_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 46, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

Order Products (Prior) (Composite PK)

In [45]:
%%sql
SELECT order_id, product_id, COUNT(*) AS duplicates
FROM order_products_prior
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 47, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>

3. Order Products (Train) (Composite PK)

In [46]:
%%sql
SELECT order_id, product_id, COUNT(*) AS duplicates
FROM order_products_train
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 48, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 3 fields>


Products (PK: product_id)

In [47]:
%%sql
SELECT product_id, COUNT(*) AS duplicates
FROM products
GROUP BY product_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 49, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

Aisles (PK: aisle_id)

In [48]:
%%sql
SELECT aisle_id, COUNT(*) AS duplicates
FROM aisles
GROUP BY aisle_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 50, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

Departments (PK: department_id)

In [49]:
%%sql
SELECT department_id, COUNT(*) AS duplicates
FROM departments
GROUP BY department_id
HAVING COUNT(*) > 1;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 51, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

#### Referential Integrity

In [50]:
%%sql
SELECT COUNT(*) AS orphan_orders
FROM order_products_prior p
LEFT JOIN orders o
ON p.order_id = o.order_id
WHERE o.order_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 52, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [51]:
%%sql
SELECT COUNT(*) AS orphan_products
FROM order_products_prior p
LEFT JOIN products pr
ON p.product_id = pr.product_id
WHERE pr.product_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 53, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [52]:
%%sql
SELECT COUNT(*) AS missing_aisles
FROM products p
LEFT JOIN aisles a
ON p.aisle_id = a.aisle_id
WHERE a.aisle_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 54, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [53]:
%%sql
SELECT COUNT(*) AS missing_departments
FROM products p
LEFT JOIN departments d
ON p.department_id = d.department_id
WHERE d.department_id IS NULL;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 55, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

#### Range Check

In [54]:
%%sql
SELECT DISTINCT order_dow FROM orders ORDER BY order_dow;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 56, Finished, Available, Finished, False)

<Spark SQL result set with 7 rows and 1 fields>

In [55]:
%%sql
SELECT DISTINCT order_hour_of_day FROM orders ORDER BY order_hour_of_day;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 57, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 1 fields>

### Exploratory Checks

#### Orders per User

In [56]:
%%sql
SELECT user_id, COUNT(*) AS orders_per_user
FROM orders
GROUP BY user_id
ORDER BY orders_per_user DESC
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 58, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

In [57]:
%%sql
SELECT user_id, COUNT(*) AS orders_per_user
FROM orders
GROUP BY user_id
ORDER BY orders_per_user
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 59, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

#### Basket Size

In [58]:
%%sql
SELECT user_id, COUNT(*) AS orders_per_user
FROM orders
GROUP BY user_id
ORDER BY orders_per_user DESC
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 60, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

In [59]:
%%sql
SELECT order_id, COUNT(product_id) AS basket_size
FROM order_products_prior
GROUP BY order_id
ORDER BY basket_size
LIMIT 20;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 61, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 2 fields>

#### Reorder Distribution

In [60]:
%%sql
SELECT reordered, COUNT(*) AS count
FROM order_products_prior
GROUP BY reordered;

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 62, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

### Creating Product Hierarchy (Denormalization Step)

#### Purpose

This step creates a **denormalized product table** by combining product metadata from multiple tables:

- `products`
- `aisles`
- `departments`



In [61]:
product_details = (
    products_clean
    .join(aisles, "aisle_id", "left")
    .join(departments, "department_id", "left")
)

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 63, Finished, Available, Finished, False)

In [62]:
product_details.show()

StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 64, Finished, Available, Finished, False)

+-------------+--------+----------+--------------------+--------------------+---------------+
|department_id|aisle_id|product_id|        product_name|               aisle|     department|
+-------------+--------+----------+--------------------+--------------------+---------------+
|           19|      61|         1|Chocolate Sandwic...|       cookies cakes|         snacks|
|           13|     104|         2|    All-Seasons Salt|   spices seasonings|         pantry|
|            7|      94|         3|Robust Golden Uns...|                 tea|      beverages|
|            1|      38|         4|Smart Ones Classi...|        frozen meals|         frozen|
|           13|       5|         5|Green Chile Anyti...|marinades meat pr...|         pantry|
|           11|      11|         6|        Dry Nose Oil|    cold flu allergy|  personal care|
|            7|      98|         7|Pure Coconut Wate...|       juice nectars|      beverages|
|            1|     116|         8|Cut Russet Potato...|    

### Write Silver Cells

In [64]:
# =========================
# NO CLEANING REQUIRED (VALIDATED DATA)
# =========================
orders_clean = orders
order_products_prior_clean = order_products_prior
order_products_train_clean = order_products_train
aisles_clean = aisles
departments_clean = departments

# =========================
# PERFORMANCE OPTIMIZATION
# =========================
order_products_prior_clean = order_products_prior_clean.repartition("order_id")
order_products_train_clean = order_products_train_clean.repartition("order_id")

# =========================
# WRITE SILVER TABLES (ALL 6 + ENRICHED)
# =========================

# Orders
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

# Order Products (Prior)
order_products_prior_clean.write.format("delta").mode("overwrite").saveAsTable("silver_order_products_prior")

# Order Products (Train)
order_products_train_clean.write.format("delta").mode("overwrite").saveAsTable("silver_order_products_train")

# Products (raw cleaned)
products_clean.write.format("delta").mode("overwrite").saveAsTable("silver_products_raw")

# Aisles
aisles_clean.write.format("delta").mode("overwrite").saveAsTable("silver_aisles")

# Departments
departments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_departments")

# Enriched Products (for downstream use)
product_details.write.format("delta").mode("overwrite").saveAsTable("silver_products")


# =========================
# FINAL VALIDATION
# =========================
print("silver_orders:", spark.table("silver_orders").count())
print("silver_order_products_prior:", spark.table("silver_order_products_prior").count())
print("silver_order_products_train:", spark.table("silver_order_products_train").count())
print("silver_products:", spark.table("silver_products").count())


StatementMeta(, ec14ebce-44cb-4a38-b280-dbc0ceef176e, 66, Finished, Available, Finished, False)

silver_orders: 3421083
silver_order_products_prior: 32434489
silver_order_products_train: 1384617
silver_products: 49688
